In [ ]:
import alc
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

In [ ]:
# Definimos una matriz 5x2 con la coordenadas de los píxeles del centro y los extremos de la H de la imagen 
img_vecs = np.array([
    [525.0, 1003.0], # 0: Centro
    [402.0, 1097.0], # 1: Extremo sup izq
    [392.0, 945.0],  # 2: Extremo sup der
    [644.0, 912.0],  # 3: Extremo inf der
    [656.0, 1061.0]  # 4: Extremo inf izq
])

# Extraemos del csv la matriz de las coordenadas en el mundo real y limpiamos el df
ruta_obj = os.path.join('heli', 'object_points.csv')
df_obj = pd.read_csv(ruta_obj, sep=';', header=0)
obj_vecs = df_obj.iloc[:, 1:4].values

# Extraemos la matriz K directamente, acá no hay que limpiar nada porque ya está limpio
ruta_K = os.path.join('heli', 'K.csv')
df_K = pd.read_csv(ruta_K, sep=';', header=None)
K = df_K.values

"""
Con todo limpio y con los tipos de datos que necesita la función aplicarPOSIT(), 
la llamamos, la función recibirá los 5 puntos de la imagen (img_vecs) y del mundo real (obj_vecs), 
la matriz de calibración K, el límite máximo de iteraciones MaxIt y la tolerancia atol.
La función retornará la matriz de Rotación R (3x3) y de traslación t (3x1)
"""
R_final, t_final = alc.aplicarPOSIT(
    obj_vecs=obj_vecs, 
    img_vecs=img_vecs, 
    K=K, 
    MaxIt=100, 
    atol=1e-6
)

print(f"Matriz R:\n{R_final}")
print("\n")
print(f"Vector t:\n{t_final}")

### Conclusiones del item a):

aplicarPOSIT() convergió antes de llegar al límite de iteraciones y como los RMS_1 y RMS_2 tienen bastante diferencia, agarró la solución con menos promedio de error. 

El resultado de t ubica al dron a 4.423m de profundidad frente al lente y desplazado lateralmente en x (-0.688m) y en y (0.905m)

La distancia euclidiana entre el dron y el centro de la H sería $||t||_2$ = 4.57m

### Siguiente item
b) o f) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

Como las conclusiones del ítem anterior mostraron que el dron está desplazado e inclinado respecto del helipuerto, buscamos una pose objetivo ($R_{obj}$ y $t_{obj}$) que lo ubique centrado y perpendicular sobre la H.

### 1. Posición del dron y traslación

Partiendo de la relación entre un punto en el sistema del mundo ($P_{real}$) y sus coordenadas en la cámara ($P_{cam}$):

$$P_{cam} = R P_{real} + t$$

Para hallar la posición del centro óptico de la cámara ($C$) expresada en el sistema de referencia del helipuerto, imponemos $P_{cam} = 0$ y despejamos $P_{real} = C$:

$$0 = R C + t$$
$$R C = -t$$
$$C = -R^{-1} t$$

Como $R$ es una matriz ortogonal ($R^{-1} = R^T$), obtenemos:

$$C = -R^T t$$

Aplicando esta fórmula tanto para la pose actual ($C_{actual}$) como para la objetivo ($C_{obj}$), ambas posiciones quedan expresadas en el mismo sistema de referencia (el del helipuerto) y podemos restarlas para obtener el desplazamiento necesario en $X$, $Y$ y $Z$:

$$\Delta C = C_{obj} - C_{actual}$$

### 2. Rotación relativa ($dR$)

Tomando dos puntos del mundo $P_1$ y $P_2$ que definen un vector de dirección $v_w = P_2 - P_1$, sus proyecciones en el sistema de la cámara son:

$$P_{c,1} = R P_1 + t$$
$$P_{c,2} = R P_2 + t$$

Al restarlos para obtener el vector de dirección visto desde la cámara, la traslación $t$ se cancela:

$$v_c = P_{c,2} - P_{c,1} = (R P_2 + t) - (R P_1 + t) = R (P_2 - P_1) = R v_w$$

En nuestro caso particular, el mismo vector del mundo se expresa en la pose actual y en la objetivo como:

$$v_{actual} = R_{final} v_w$$
$$v_{obj} = R_{obj} v_w$$

La matriz de rotación relativa $dR$ debe transformar las coordenadas del vector desde la pose actual hacia la pose objetivo:

$$v_{obj} = dR v_{actual}$$

Sustituyendo:

$$R_{obj} v_w = dR (R_{final} v_w) = (dR R_{final}) v_w$$

Como esta igualdad vale para todo vector $v_w$, se cumple que $dR R_{final} = R_{obj}$. Despejando y conociendo la ortogonalidad de $R_{final}$ ($R_{final}^{-1} = R_{final}^T$), llegamos a:

$$dR = R_{obj} R_{final}^T$$

$R_{obj}$: Se fija como la matriz ${diag}(1, -1, -1)$ para que el eje óptico de la cámara apunte perpendicularmente hacia el piso con $Z = -1$ y el determinante se mantenga en $+1$

$t_{obj}$: Se toma como altura de referencia la distancia euclidiana actual entre la cámara y el centro de la H fijando $t_{obj}=(0,0,h)^T$ para centrar la H en el centro de la cámara

In [ ]:
# R objetico con la cámara mirando perpendicularmente hacia abajo
R_obj_ej_6 = np.array([[1.0, 0, 0], [0, -1.0, 0], [0, 0, -1.0]])

# t objetivo centrado en x e y, con z igual a la distancia 
# euclidiana entre el dron y el centro de la H
h_ej_6 = np.sqrt(np.sum(t_final**2))
t_obj_ej_6 = np.array([0, 0, h_ej_6])

# Posición actual (C = -R^t * t)
C_actual_ej_6 = (-1) * alc.arroba(alc.transpuesta(R_final), t_final)
# Posición objetivo (C = -R^t * t)
C_obj_ej_6 = (-1) * alc.arroba(alc.transpuesta(R_obj_ej_6), t_obj_ej_6)
# Delta_C final 
delta_C = C_obj_ej_6 - C_actual_ej_6

# La rotación relativa dR = R_obj * R_final^T
dR_ej_6 = alc.arroba(R_obj_ej_6, alc.transpuesta(R_final))

print("R_obj:\n", R_obj_ej_6)
print("t_obj:\n", t_obj_ej_6)

print("Traslación que tiene que hacer el drone:", delta_C)
print("Rotación que tiene que hacer: dR:\n", dR_ej_6)

#### Conclusiones del ítem f)
Al pasar ambas poses al sistema del helipuerto, se obtiene que el dron debe desplazarse aproximadamente -0.65 m en X, -0.94 m en Y y 0.15 m en Z, aplicando también la rotación dR para corregir su inclinación.